# JS08 — Lab 1: Hierarchical Clustering (Agglomerative)

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membuat model klasterisasi hierarki *bottom-up* (Agglomerative) untuk mengelompokkan customer toko ritel berdasarkan `Annual Income` dan `Spending Score`, serta mengevaluasinya dengan *Silhouette Score*.

**Dataset:** `Mall_Customers.csv` (200 baris) — diunduh otomatis oleh cell di bawah, atau unduh manual dari modul lalu unggah ke Colab.


# Lab 1 — Hierarchical Clustering

Pada praktikum ini dibuat model clustering hierarki (bottom-up / *agglomerative*) untuk mengelompokkan customer toko ritel berdasarkan `Annual Income` dan `Spending Score`.

## Langkah 0 — Unduh Dataset

Dataset: `Mall_Customers.csv` (200 baris, 5 kolom).

Cell di bawah akan memakai file lokal bila sudah ada; kalau belum, file diunduh otomatis dari mirror publik. Alternatif: unduh manual dari halaman modul, lalu unggah ke Colab (`Files` ▸ *Upload*).

In [ ]:
DATASET_URL = (
    "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/"
    "Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/"
    "Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv"
)
DATASET_PATH = "Mall_Customers.csv"

from pathlib import Path

if not Path(DATASET_PATH).exists():
    try:
        from urllib.request import urlretrieve
        urlretrieve(DATASET_URL, DATASET_PATH)
        print(f"Dataset diunduh: {DATASET_PATH}")
    except Exception as e:
        print("Gagal mengunduh otomatis:", e)
        try:  # khusus Google Colab
            from google.colab import files
            print("Silakan unggah Mall_Customers.csv secara manual:")
            files.upload()
        except ImportError:
            pass

print("File tersedia:", Path(DATASET_PATH).exists())

## Langkah 1 — Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Langkah 2 — Load Dataset

Load dataset lalu lakukan inspeksi terhadap data.

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_PATH)

# Inspect info
df.info()

In [ ]:
# Inspeksi awal data
df.head()

In [ ]:
# Statistik deskriptif
df.describe()

## Langkah 3 — Slicing Fitur

Hanya fitur `Annual Income (k$)` dan `Spending Score (1-100)` yang dipakai.

In [ ]:
# Slicing - Select Annual Income and Spending Score only
X = df.iloc[:, [3, 4]].values  # .values -> ndarray
X[:5]

## Langkah 4 — Visualisasi Dendrogram

Nilai **threshold** pada sumbu-y (Euclidean distance) dipakai untuk menentukan jumlah cluster. Dari dendrogram terlihat 3 cluster utama, tetapi dengan threshold rendah (100–150) bisa dipilih 5 cluster.

In [ ]:
# Build dendrogram
import scipy.cluster.hierarchy as sch

# Build dendrogram using ward distance
# more info - https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
dendrogram = sch.dendrogram(sch.linkage(X, method='ward'))

# Plot
plt.title('Dendrogram')
plt.xlabel('Customers')
plt.ylabel('Euclidean distances')
plt.show()

## Langkah 5 — Membuat Model Agglomerative

Model *bottom-up* dengan 5 cluster, `linkage='ward'` dan jarak Euclidean.

In [ ]:
# Build Cluster using Agglomerative Method
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')
y_hc = hc.fit_predict(X)

np.unique(y_hc, return_counts=True)

## Langkah 6 — Visualisasi Cluster

In [ ]:
# Visualize The Result
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s=100, c='red',     label='Cluster 1')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s=100, c='blue',    label='Cluster 2')
plt.scatter(X[y_hc == 2, 0], X[y_hc == 2, 1], s=100, c='green',   label='Cluster 3')
plt.scatter(X[y_hc == 3, 0], X[y_hc == 3, 1], s=100, c='cyan',    label='Cluster 4')
plt.scatter(X[y_hc == 4, 0], X[y_hc == 4, 1], s=100, c='magenta', label='Cluster 5')
plt.title('Clusters of customers')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.show()

## Langkah 7 — Evaluasi Hasil Cluster

Evaluasi dengan *silhouette coefficient*.

In [ ]:
from sklearn.metrics import silhouette_score

# Evaluate the clustering using silhouette coefficient
silhouette_avg = silhouette_score(X, y_hc)
print(f'Silhouette Coefficient: {silhouette_avg}')

**Catatan modul:** nilai *silhouette* sekitar **0.553** tergolong cukup baik. Modul meminta eksperimen jumlah cluster — lihat cell tambahan di bawah.

### [tambahan] Perbandingan jumlah cluster (k = 3..6)

Membantu menjawab pertanyaan modul: jumlah cluster mana yang paling baik menurut *Silhouette Score*?

In [ ]:
from sklearn.metrics import davies_bouldin_score

rows = []
for k in range(3, 7):
    y = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward').fit_predict(X)
    rows.append({
        'n_clusters': k,
        'silhouette': silhouette_score(X, y),
        'dbi': davies_bouldin_score(X, y),
    })

hasil_lab1 = pd.DataFrame(rows).set_index('n_clusters').round(4)
hasil_lab1

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(hasil_lab1.index, hasil_lab1['silhouette'], marker='o', color='tab:blue')
ax1.set_xlabel('Jumlah cluster (k)')
ax1.set_ylabel('Silhouette Score', color='tab:blue')
ax1.set_xticks(hasil_lab1.index)

ax2 = ax1.twinx()
ax2.plot(hasil_lab1.index, hasil_lab1['dbi'], marker='s', color='tab:red')
ax2.set_ylabel('Davies-Bouldin Index', color='tab:red')

plt.title('Pemilihan jumlah cluster Agglomerative')
plt.tight_layout()
plt.show()

---
## Kesimpulan Lab 1

**Jumlah cluster terbaik & alasannya:** TBD

**Nilai Silhouette Score model (k=5):** TBD

*(Isi dengan angka dari output di atas. Silhouette ≈ 0.55 = struktur cluster cukup baik; bandingkan dengan hasil cell [tambahan] untuk k lain.)*